# **FastAPI**

This assignment demonstrates how to build a simple Product Store API using FastAPI.  
The API allows us to manage products, filter them by category, check stock availability, search products, and view store summaries.

Technologies Used:
- Python
- FastAPI
- Uvicorn
- Jupyter Notebook

# **Question 1: Filter Products by Minimum Price**

In FastAPI, query parameters allow clients to filter results using URL parameters.

Example:
GET /products/filter?min_price=400

This endpoint filters products whose price is greater than or equal to the given minimum price.

FastAPI provides Query() to define optional query parameters with validation and description.

In [20]:
from fastapi import FastAPI, Query
from typing import Optional

app = FastAPI()

products = [
    {"id":1,"name":"Wireless Mouse","price":499,"category":"Electronics","in_stock":True},
    {"id":2,"name":"Notebook","price":99,"category":"Stationery","in_stock":True},
    {"id":3,"name":"USB Hub","price":799,"category":"Electronics","in_stock":False},
    {"id":4,"name":"Pen Set","price":49,"category":"Stationery","in_stock":True}
]

@app.get("/products/filter")
def filter_products(
    category: Optional[str] = None,
    max_price: Optional[int] = None,
    min_price: Optional[int] = Query(None, description="Minimum price")
):

    result = products

    if category:
        result = [p for p in result if p["category"].lower() == category.lower()]

    if max_price:
        result = [p for p in result if p["price"] <= max_price]

    if min_price:
        result = [p for p in result if p["price"] >= min_price]

    return result

# **Question 2 — Product Price Endpoint**
Path parameters allow retrieving specific data using an identifier.

Example:
GET /products/1/price

This endpoint returns only the name and price of the product instead of the full product information.

In [22]:
@app.get("/products/{product_id}/price")
def get_product_price(product_id: int):

    for product in products:
        if product["id"] == product_id:
            return {
                "name": product["name"],
                "price": product["price"]
            }

    return {"error": "Product not found"}

# **Question 3 — Customer Feedback (Pydantic + POST)**

Pydantic models are used in FastAPI for request validation.

In this task, we create a CustomerFeedback model to ensure:

customer_name → minimum 2 characters  
product_id → must be greater than 0  
rating → between 1 and 5  
comment → optional text

POST requests are tested using Swagger UI.

In [24]:
from pydantic import BaseModel, Field
from typing import Optional

feedback = []

class CustomerFeedback(BaseModel):
    customer_name: str = Field(..., min_length=2, max_length=100)
    product_id: int = Field(..., gt=0)
    rating: int = Field(..., ge=1, le=5)
    comment: Optional[str] = Field(None, max_length=300)

@app.post("/feedback")
def submit_feedback(data: CustomerFeedback):

    feedback.append(data.dict())

    return {
        "message": "Feedback submitted successfully",
        "feedback": data.dict(),
        "total_feedback": len(feedback)
    }

# **Question 4 — Product Summary Dashboard**
This endpoint provides business insights for products.

It calculates:
- total number of products
- number of products in stock
- number of products out of stock
- most expensive product
- cheapest product
- list of unique product categories

Python built-in functions like max(), min(), and set() help compute these values.

In [26]:
@app.get("/products/summary")
def product_summary():

    in_stock = [p for p in products if p["in_stock"]]
    out_stock = [p for p in products if not p["in_stock"]]

    expensive = max(products, key=lambda p: p["price"])
    cheapest = min(products, key=lambda p: p["price"])

    categories = list(set(p["category"] for p in products))

    return {
        "total_products": len(products),
        "in_stock_count": len(in_stock),
        "out_of_stock_count": len(out_stock),
        "most_expensive": {
            "name": expensive["name"],
            "price": expensive["price"]
        },
        "cheapest": {
            "name": cheapest["name"],
            "price": cheapest["price"]
        },
        "categories": categories
    }

# **Question 5 — Bulk Order API**

This endpoint processes bulk orders from companies.

It validates:
- company name
- contact email
- list of items

Each item includes:
product_id
quantity

The API checks if the product exists and if it is in stock.

If successful:
subtotal = price × quantity

The API returns:
confirmed orders
failed orders
grand total price

In [28]:
from typing import List

class OrderItem(BaseModel):
    product_id: int = Field(..., gt=0)
    quantity: int = Field(..., gt=0, le=50)

class BulkOrder(BaseModel):
    company_name: str = Field(..., min_length=2)
    contact_email: str = Field(..., min_length=5)
    items: List[OrderItem] = Field(..., min_items=1)

@app.post("/orders/bulk")
def place_bulk_order(order: BulkOrder):

    confirmed = []
    failed = []
    grand_total = 0

    for item in order.items:

        product = next((p for p in products if p["id"] == item.product_id), None)

        if not product:
            failed.append({
                "product_id": item.product_id,
                "reason": "Product not found"
            })

        elif not product["in_stock"]:
            failed.append({
                "product_id": item.product_id,
                "reason": f"{product['name']} is out of stock"
            })

        else:
            subtotal = product["price"] * item.quantity
            grand_total += subtotal

            confirmed.append({
                "product": product["name"],
                "qty": item.quantity,
                "subtotal": subtotal
            })

    return {
        "company": order.company_name,
        "confirmed": confirmed,
        "failed": failed,
        "grand_total": grand_total
    }

# **Server Running Code**

In [36]:
import threading
import uvicorn

def run_server():
    uvicorn.run(app, host="127.0.0.1", port=800)

# Start server in a separate thread
thread = threading.Thread(target=run_server, daemon=True)
thread.start()

INFO:     Started server process [11608]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:800 (Press CTRL+C to quit)
